# 🎓 Using a Database with Flask
### Unit 2, Module 03 — MSc (IT) Hands-On Python

Welcome! In this notebook, you will learn how to connect a database to your Flask web application **step by step**.

---

### 🤔 Why do we need a database?
If you store data in a Python list like `students = []`, what happens when you restart Flask?
👉 **Everything is wiped out and lost!**

A **database** saves your data permanently on your computer in a file. Even if you restart your server or turn off your computer, your data stays safe!

---

### 💡 The Big Picture: What is an ORM?
Instead of writing complicated SQL queries by hand, we use **Flask-SQLAlchemy** (an ORM).  
It allows you to treat database tables just like normal Python classes:

- **Table** $
 A Python Class (`class Student`)
- **Row** $
 A Python Object (`student1 = Student(...)`)
- **Column** $
 A Variable (`name`, `course`, `grade`)

Let's begin!


---
## 🔍 Quick Example: Direct SQLite Without ORM (Raw SQL)

Before using Flask-SQLAlchemy, let's see how Python talks to SQLite directly using the built-in `sqlite3` library in just 5 lines:


In [ ]:
# Quick Example: Direct SQLite in Python (No ORM)
import sqlite3

# 1. Connect to SQLite database (creates an in-memory database)
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# 2. Create table using raw SQL
cursor.execute("CREATE TABLE students (id INTEGER PRIMARY KEY, name TEXT, course TEXT)")

# 3. Insert a student using ? placeholder (prevents SQL injection)
cursor.execute("INSERT INTO students (name, course) VALUES (?, ?)", ("Aarav Sharma", "Python"))
conn.commit()

# 4. Fetch and display
cursor.execute("SELECT * FROM students")
print("Direct SQLite Result:", cursor.fetchall())
conn.close()

# 💡 Notice: Writing raw SQL strings everywhere can get tedious in big apps.
# That's why we use Flask-SQLAlchemy (ORM)!


---
## ⚙️ Step 1: Connect Flask to the Database (Just 4 Lines!)

To use Flask-SQLAlchemy, we only need 4 simple lines:
1. Import `Flask` and `SQLAlchemy`.
2. Create our Flask `app`.
3. Set the database location: `'sqlite:///students.db'`.
4. Create `db = SQLAlchemy(app)`.

Let's run the setup:


In [ ]:
# Step 1: Initialize Flask and Database
from flask import Flask, request
from flask_sqlalchemy import SQLAlchemy

app = Flask(__name__)

# Use an in-memory database for clean, fast classroom testing
# (To save to a real file, use: 'sqlite:///students.db')
app.config['SQLALCHEMY_DATABASE_URI'] = 'sqlite:///:memory:'
app.config['SQLALCHEMY_TRACK_MODIFICATIONS'] = False

# Connect SQLAlchemy to Flask
db = SQLAlchemy(app)

print("✅ Step 1 Complete: Database successfully connected to Flask!")


---
## 📝 Step 2: Define the Student Table (Model)

Now we define what a `Student` looks like. In Flask-SQLAlchemy, we create a class that inherits from `db.Model`:

- `id`: Unique roll number (1, 2, 3...), marked as `primary_key=True`.
- `name`: Student's name (Text), marked as `nullable=False` (cannot be empty).
- `course`: Enrolled course (Text).
- `grade`: Letter grade like 'A' or 'B'.

Let's define the class:


In [ ]:
# Step 2: Define the Student Model
class Student(db.Model):
    id = db.Column(db.Integer, primary_key=True)       # Unique ID
    name = db.Column(db.String(100), nullable=False)   # Name cannot be blank
    course = db.Column(db.String(100), nullable=False) # Course name
    grade = db.Column(db.String(5))                    # Grade (e.g. 'A+', 'B')

    def __repr__(self):
        return f"<Student #{self.id}: {self.name} ({self.course}) - Grade: {self.grade}>"

print("✅ Step 2 Complete: Student model defined!")


---
## 🔨 Step 3: Tell the Database to Create the Table (`db.create_all()`)

Now we tell SQLAlchemy to actually create the `students` table in SQLite.  
We wrap this inside `with app.app_context():` so SQLAlchemy knows which Flask app to use:


In [ ]:
# Step 3: Create the table
with app.app_context():
    db.create_all()

print("✅ Step 3 Complete: 'students' table is created and ready for data!")


---
## ➕ Step 4: CREATE — Adding Your First Student

Adding data to the database takes **2 simple steps**:
1. **`db.session.add(student)`**: Puts the student into your shopping cart (staging).
2. **`db.session.commit()`**: Clicks "checkout" to save permanently to disk!

Notice how SQLite automatically gives our student an `id` of `1`:


In [ ]:
# Step 4: Add one student
with app.app_context():
    # 1. Create a Student object
    s1 = Student(name="Aarav Sharma", course="Python", grade="A")

    # 2. Stage and save
    db.session.add(s1)
    db.session.commit()

    print(f"✅ Step 4 Complete: Student saved! Auto-assigned ID is: {s1.id}")


---
## 📦 Step 5: Adding Multiple Students at Once (`add_all`)

If you have several students, you don't need to commit one by one. Use `db.session.add_all([...])`:


In [ ]:
# Step 5: Add multiple students
with app.app_context():
    s2 = Student(name="Diya Patel", course="Python", grade="A+")
    s3 = Student(name="Rohan Verma", course="Java", grade="B")
    s4 = Student(name="Ananya Iyer", course="Data Science", grade="A")

    db.session.add_all([s2, s3, s4])
    db.session.commit()

    print("✅ Step 5 Complete: Added Diya, Rohan, and Ananya!")


---
## 📋 Step 6: READ — Fetching All Students (`all()`)

To see all students in your database, call `Student.query.all()`.  
It returns a regular Python list that you can loop through:


In [ ]:
# Step 6: View all students
with app.app_context():
    students = Student.query.all()
    
    print(f"Total students in database: {len(students)}\n")
    for s in students:
        print(f"• ID: {s.id} | Name: {s.name:<15} | Course: {s.course:<12} | Grade: {s.grade}")

print("\n✅ Step 6 Complete: Retrieved all student records!")


---
## 🎯 Step 7: READ — Finding One Student by ID (`db.session.get`)

If you know a student's ID (like ID #2), you can look them up instantly using `db.session.get(Student, id)`:


In [ ]:
# Step 7: Find student by ID
with app.app_context():
    student = db.session.get(Student, 2)
    print("Search result for ID #2:")
    print(f"➔ Found: {student.name}, enrolled in {student.course} with grade {student.grade}")

print("\n✅ Step 7 Complete: Found student by primary key ID!")


---
## 🔍 Step 8: READ — Searching & Filtering (`filter_by`)

What if you only want students enrolled in **Python**?  
Use `.filter_by(course="Python").all()`:


In [ ]:
# Step 8: Filter by course
with app.app_context():
    python_students = Student.query.filter_by(course="Python").all()

    print("Students taking Python:")
    for s in python_students:
        print(f"• {s.name} (Grade: {s.grade})")

print("\n✅ Step 8 Complete: Filtered query executed successfully!")


---
## ✏️ Step 9: UPDATE — Changing a Student's Details

Updating a student in Flask-SQLAlchemy is as easy as changing a Python variable:
1. Fetch the student: `student = db.session.get(Student, 3)`
2. Change the value: `student.grade = "A"`
3. Save the change: `db.session.commit()`

Let's upgrade Rohan's grade from `B` to `A`:


In [ ]:
# Step 9: Update a record
with app.app_context():
    rohan = db.session.get(Student, 3)
    print(f"Before update: {rohan.name}'s grade was {rohan.grade}")

    # Change the grade and save
    rohan.grade = "A"
    db.session.commit()

    print(f"After update : {rohan.name}'s grade is now {rohan.grade}")

print("✅ Step 9 Complete: Database record updated!")


---
## 🗑️ Step 10: DELETE — Removing a Student

To delete a record:
1. Find the student: `student = db.session.get(Student, 4)`
2. Mark for deletion: `db.session.delete(student)`
3. Save: `db.session.commit()`

Let's remove student #4 and verify that only 3 students remain:


In [ ]:
# Step 10: Delete a record
with app.app_context():
    to_delete = db.session.get(Student, 4)
    print(f"Deleting: {to_delete.name}")

    db.session.delete(to_delete)
    db.session.commit()

    remaining_count = Student.query.count()
    print(f"Remaining students in database: {remaining_count}")

print("✅ Step 10 Complete: Student successfully deleted!")


---
## 🌐 Step 11: Using the Database in a Real Flask Web Route!

Now let's connect our database to a real web page!  
When a user visits `http://127.0.0.1:5000/students` in their browser, Flask queries the database and displays the student list:


In [ ]:
# Step 11: Web route that shows students
@app.route('/students')
def show_students():
    all_students = Student.query.all()
    
    html = "<h1>🎓 Student Roster</h1><ul>"
    for s in all_students:
        html += f"<li><strong>{s.name}</strong> — {s.course} (Grade: {s.grade})</li>"
    html += "</ul>"
    return html

# Let's test the route right here using Flask's test client!
with app.test_client() as client:
    response = client.get('/students')
    print("HTML response generated by Flask route:")
    print(response.data.decode('utf-8'))

print("✅ Step 11 Complete: Flask route successfully fetched data from SQLite!")


---
## 📥 Step 12: Adding Data from an HTML Form (POST)

When a student fills out a registration form and clicks **Submit**, the form sends a `POST` request to Flask.  
Inside our route, we read the form values using `request.form.get()` and save them to the database:


In [ ]:
# Step 12: Web route to add a student
@app.route('/students/add', methods=['POST'])
def add_student_route():
    # Read values sent from the form
    name = request.form.get('name')
    course = request.form.get('course')
    grade = request.form.get('grade', 'N/A')

    # Save to database
    new_student = Student(name=name, course=course, grade=grade)
    db.session.add(new_student)
    db.session.commit()

    return f"🎉 Success! Added student: {name} (ID #{new_student.id})"

# Simulate a user submitting the form!
with app.test_client() as client:
    response = client.post('/students/add', data={
        'name': 'Pooja Hegde',
        'course': 'Machine Learning',
        'grade': 'A+'
    })
    print(response.data.decode('utf-8'))

# Verify Pooja is now in the database
with app.app_context():
    pooja = Student.query.filter_by(name='Pooja Hegde').first()
    print(f"Verified in database: {pooja}")

print("\n✅ Step 12 Complete: Form submission successfully saved to database!")


---
## 📖 Quick Summary & Cheat Sheet

Here are the 4 commands you will use 90% of the time in Flask:

| CRUD Action | What You Write in Python | What it Does in SQL |
|---|---|---|
| **C**reate | `db.session.add(student)`<br>`db.session.commit()` | `INSERT INTO students ...` |
| **R**ead (All) | `Student.query.all()` | `SELECT * FROM students` |
| **R**ead (By ID) | `db.session.get(Student, 1)` | `SELECT * FROM students WHERE id = 1` |
| **R**ead (Filter) | `Student.query.filter_by(course='Python').all()` | `SELECT * FROM students WHERE course = 'Python'` |
| **U**pdate | `student.grade = 'A+'`<br>`db.session.commit()` | `UPDATE students SET grade = 'A+' WHERE id = ...` |
| **D**elete | `db.session.delete(student)`<br>`db.session.commit()` | `DELETE FROM students WHERE id = ...` |

> ⚠️ **Golden Rule for Beginners:**  
> Whenever you **Add**, **Update**, or **Delete**, always remember to call:  
> 👉 `db.session.commit()`!  
> If you forget to commit, your changes will **not** be saved!


---
## 🏋️ Step 13: Try It Yourself! (5-Minute Lab Challenge)

Now it's your turn to write code!

### Your Task:
1. Define a `Book` model with 3 columns:
   - `id`: Integer primary key
   - `title`: String (not null)
   - `price`: Float (not null)
2. Create the table using `db.create_all()`.
3. Add a book named **"Python Crash Course"** with price **29.99**.
4. Query all books and print them.

Run the code cell below to test your solution!


In [ ]:
# Step 13: Student Exercise Code
# 1. Define the Book model
class Book(db.Model):
    id = db.Column(db.Integer, primary_key=True)
    title = db.Column(db.String(150), nullable=False)
    price = db.Column(db.Float, nullable=False)

# 2. Create the table
with app.app_context():
    db.create_all()

    # 3. Add a book
    book1 = Book(title="Python Crash Course", price=29.99)
    db.session.add(book1)
    db.session.commit()

    # 4. Query and print
    all_books = Book.query.all()
    print("📚 Books in Library Database:")
    for b in all_books:
        print(f"• #{b.id}: {b.title} - ${b.price:.2f}")

print("\n🎉 Congratulations! You have mastered using databases with Flask!")
